# 任务 C：vLLM 本地部署与性能测试

## 学习目标
- 理解模型部署的基本概念
- 学会在 GPU 服务器上用 vLLM 部署开源模型
- 对比云端 API 与本地部署的优劣

## vLLM 简介

**vLLM** 是一个高性能的 LLM 推理引擎，核心优势：
- **PagedAttention**：类似操作系统分页内存的技术，显存利用率提升 4 倍
- **高吞吐**：批量处理请求，比原生 HuggingFace 推理快 24 倍
- **OpenAI 兼容接口**：部署后可以直接用 OpenAI SDK 调用，和 agicto API 用法一致

## 任务流程
1. 测试 vLLM 本地服务连通性
2. 性能对比实验 — vLLM 本地 vs agicto 云端
3. 可视化对比与模型选型讨论

In [4]:
# 安装依赖（首次运行时执行）
import sys
import subprocess

def pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# 任务 C 需要的包
pip_install("openai")
pip_install("matplotlib")

print("依赖安装完成！")

依赖安装完成！


---
## 步骤 1：配置 API 客户端

同时配置两个客户端：
- **vLLM 本地服务**：运行在 GPU 服务器上，端口 8000
- **agicto 云端 API**：使用 `qwen-plus` 模型

In [5]:
import os
import time
from openai import OpenAI
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── 配置 vLLM 本地服务（云端 GPU 服务器）──
# 如果需要通过 SSH 隧道访问（更安全），先在终端执行：
#   ssh -p 23 -L 8000:localhost:8000 myuser@117.50.189.94
# 然后改回: http://localhost:8000/v1
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://117.50.189.94:8000/v1")
VLLM_MODEL = os.environ.get("VLLM_MODEL", "Qwen/Qwen3-8B")

# ── 配置 agicto 云端 API ──
API_KEY = os.environ.get("AGICTO_API_KEY", "sk-Fv9s4DBlAIpSgVdNdDrjFcri7N7hOMy7PzREWbPmbGqf7eVX")
BASE_URL = "https://api.agicto.cn/v1"
MODEL = "qwen-plus"

vllm_client = OpenAI(api_key="vllm-local", base_url=VLLM_BASE_URL)
cloud_client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print(f"vLLM 地址: {VLLM_BASE_URL}")
print(f"vLLM 模型: {VLLM_MODEL}")
print(f"云端地址: {BASE_URL}")
print(f"云端模型: {MODEL}")
print("配置完成！")

vLLM 地址: http://117.50.189.94:8000/v1
vLLM 模型: Qwen/Qwen3-8B
云端地址: https://api.agicto.cn/v1
云端模型: qwen-plus
配置完成！


---
## 步骤 2：测试 vLLM 本地服务连通性

定义通用对话函数，测试本地 vLLM 服务是否能正常响应。

In [7]:
def test_single_request(client, model, prompt, label):
    """测试单次请求"""
    print(f"\n--- 测试 {label} ---")
    print(f"Prompt: {prompt[:80]}...")
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5,
            max_tokens=512,
        )
        elapsed = time.time() - start
        content = response.choices[0].message.content
        print(f"耗时: {elapsed:.2f}s")
        print(f"输出长度: {len(content)} 字符")
        print(f"回答（前200字）: {content[:200]}...")
        return elapsed, content
    except Exception as e:
        print(f"请求失败: {e}")
        return None, None


print("=" * 60)
print("  步骤1：测试 vLLM 本地服务")
print("=" * 60)

test_single_request(
    vllm_client,
    VLLM_MODEL,
    "你好，请介绍一下番茄种植的基本要点。",
    "vLLM 本地服务",
)

  步骤1：测试 vLLM 本地服务

--- 测试 vLLM 本地服务 ---
Prompt: 你好，请介绍一下番茄种植的基本要点。...
耗时: 15.38s
输出长度: 842 字符
回答（前200字）: <think>
嗯，用户让我介绍一下番茄种植的基本要点。首先，我需要确定用户的需求是什么。可能他们是一个刚开始种植番茄的新手，或者想提高自己的种植技术。番茄种植涉及很多方面，比如品种选择、土壤准备、播种育苗、移栽、田间管理、病虫害防治和收获等。

首先，我应该从品种选择开始，因为不同的品种适合不同的种植环境。比如，番茄有早熟、中熟、晚熟之分，还有适合温室和露天种植的类型。用户可能需要根据自己的种植...


(15.383566856384277,
 '<think>\n嗯，用户让我介绍一下番茄种植的基本要点。首先，我需要确定用户的需求是什么。可能他们是一个刚开始种植番茄的新手，或者想提高自己的种植技术。番茄种植涉及很多方面，比如品种选择、土壤准备、播种育苗、移栽、田间管理、病虫害防治和收获等。\n\n首先，我应该从品种选择开始，因为不同的品种适合不同的种植环境。比如，番茄有早熟、中熟、晚熟之分，还有适合温室和露天种植的类型。用户可能需要根据自己的种植条件来选择合适的品种。\n\n接下来是土壤准备，番茄喜欢排水良好、富含有机质的土壤。可能需要提到土壤pH值，通常在6.0到7.0之间比较合适。用户可能不太清楚如何测试土壤pH，或者如何改良土壤，比如添加腐熟的有机肥。\n\n然后是播种育苗，这里需要说明播种时间，比如根据当地气候，一般在最后一次霜冻后播种。育苗的容器和基质也很重要，可能需要提醒用户使用疏松透气的介质，避免使用过湿的土壤导致烂苗。\n\n移栽是另一个关键步骤，要提到移栽前的炼苗，以及移栽时的注意事项，比如带土移栽、适当深度等。用户可能不知道移栽后的管理，比如遮阴、浇水等。\n\n田间管理包括搭架、整枝、疏果和施肥。这部分可能需要详细说明，比如搭架的方式，整枝的不同方法（单干、双干），疏果的重要性，以及施肥的时机和种类。\n\n病虫害防治部分，用户可能关心如何预防和处理常见的病虫害，比如晚疫病、蚜虫等。需要提到预防措施和有机或化学防治方法，同时强调预防的重要性。\n\n最后是收获和储存，要说明最佳收获时间，以及储存条件，比如温度和湿度控制。用户可能希望知道如何延长番茄的保存时间。\n\n还要考虑用户可能的误区，比如过度浇水、忽视病虫害防治、不及时疏果等。需要提醒他们注意这些常见问题。\n\n另外，用户可能没有提到的深层需求是希望种植番茄能获得高产和优质的果实，所以需要强调科学管理的重要性，以及如何通过合理管理提高产量和品质。可能还需要提到一些实用的小技巧，比如使用滴灌系统、轮作等。\n\n最后，要确保信息结构清晰，分点列出，便于用户理解和')

---
## 步骤 3：性能对比 — vLLM 本地 vs agicto 云端

定义基准测试函数，用同一个农业问题分别测试两种部署方式，每种跑 3 轮取平均值。

In [8]:
def benchmark_api(client, model, prompt, label, n_trials=3):
    """测试 API 的响应速度和输出质量"""
    times = []
    outputs = []

    for i in range(n_trials):
        print(f"  {label} Trial {i + 1}...")
        elapsed, content = test_single_request(client, model, prompt, label)
        if elapsed is not None:
            times.append(elapsed)
            outputs.append(content)

    if not times:
        return None

    avg_time = sum(times) / len(times)
    output_len = sum(len(o) for o in outputs) / len(outputs)

    return {
        "label": label,
        "avg_time": avg_time,
        "avg_output_len": output_len,
        "sample": outputs[0][:200] if outputs else "",
    }


print("=" * 60)
print("  步骤2：性能对比测试")
print("=" * 60)

test_prompt = (
    "请详细介绍番茄早疫病的发病原因、症状识别和防治方法，"
    "包括农业防治和化学防治的具体措施。"
)

print("\n正在测试 vLLM 本地部署...")
vllm_result = benchmark_api(vllm_client, VLLM_MODEL, test_prompt, "vLLM 本地")

print("\n正在测试 agicto 云端 API...")
cloud_result = benchmark_api(
    cloud_client, MODEL, test_prompt, "agicto 云端"
)

if vllm_result and cloud_result:
    print(f"\nvLLM 本地 — 平均响应: {vllm_result['avg_time']:.2f}s, 平均输出: {vllm_result['avg_output_len']:.0f} 字符")
    print(f"agicto 云端 — 平均响应: {cloud_result['avg_time']:.2f}s, 平均输出: {cloud_result['avg_output_len']:.0f} 字符")
else:
    print("测试失败，请检查 vLLM 服务是否已启动以及网络连接。")

  步骤2：性能对比测试

正在测试 vLLM 本地部署...
  vLLM 本地 Trial 1...

--- 测试 vLLM 本地 ---
Prompt: 请详细介绍番茄早疫病的发病原因、症状识别和防治方法，包括农业防治和化学防治的具体措施。...
耗时: 14.85s
输出长度: 834 字符
回答（前200字）: <think>
嗯，用户让我详细介绍番茄早疫病的发病原因、症状识别和防治方法，包括农业防治和化学防治的具体措施。首先，我需要确认自己对这个病害的了解是否足够。番茄早疫病，全称应该是番茄早疫病，属于真菌性病害，对吧？可能由Alternaria alternata引起，或者还有其他菌种？需要查证一下。

接下来，发病原因部分，应该包括病原菌的特性、传播途径、环境因素，比如温度、湿度，还有寄主植物的抗性...
  vLLM 本地 Trial 2...

--- 测试 vLLM 本地 ---
Prompt: 请详细介绍番茄早疫病的发病原因、症状识别和防治方法，包括农业防治和化学防治的具体措施。...
耗时: 14.98s
输出长度: 825 字符
回答（前200字）: <think>
嗯，用户让我详细介绍番茄早疫病的发病原因、症状识别和防治方法，包括农业防治和化学防治的具体措施。首先，我需要确认自己对番茄早疫病的了解是否足够。记得早疫病是由真菌引起的，主要病原体是Alternaria solani。不过可能还有其他相关病原体，比如Alternaria alternata，需要查证一下。

接下来，发病原因部分，应该包括病原菌的传播途径，比如通过气流、雨水、农具传...
  vLLM 本地 Trial 3...

--- 测试 vLLM 本地 ---
Prompt: 请详细介绍番茄早疫病的发病原因、症状识别和防治方法，包括农业防治和化学防治的具体措施。...
耗时: 15.53s
输出长度: 884 字符
回答（前200字）: <think>
嗯，用户让我详细介绍番茄早疫病的发病原因、症状识别和防治方法，包括农业和化学防治的具体措施。首先，我需要确认自己对这个病害的了解是否全面。番茄早疫病是由Alternaria alternata引起的，属于真菌病害。发病原因可能包括环境因素、管理措施不当等。

接下来是症状识别部分。早疫病在不同生长阶段的表现可能不同，

---
## 步骤 4：可视化对比

生成两张柱状图：
1. **响应速度对比** — 平均响应时间
2. **回答详细程度对比** — 平均输出字符数

同时打印两种服务生成的样例回答，对比内容质量。

In [ ]:
if vllm_result and cloud_result:
    print("=" * 60)
    print("  步骤3：生成对比图")
    print("=" * 60)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    labels = [vllm_result["label"], cloud_result["label"]]
    times = [vllm_result["avg_time"], cloud_result["avg_time"]]
    colors = ["#4e79a7", "#e15759"]

    # 图1：响应速度
    bars = ax1.bar(labels, times, color=colors, edgecolor="black")
    for bar, t in zip(bars, times):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            t + 0.1,
            f"{t:.2f}s",
            ha="center",
            fontsize=12,
            fontweight="bold",
        )
    ax1.set_ylabel("Avg Response Time (s)")
    ax1.set_title("Response Speed Comparison")
    ax1.grid(True, alpha=0.3, axis="y")

    # 图2：输出长度
    output_lens = [vllm_result["avg_output_len"], cloud_result["avg_output_len"]]
    bars = ax2.bar(labels, output_lens, color=colors, edgecolor="black")
    for bar, l in zip(bars, output_lens):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            l + 10,
            f"{l:.0f} chars",
            ha="center",
            fontsize=12,
            fontweight="bold",
        )
    ax2.set_ylabel("Avg Output Length (chars)")
    ax2.set_title("Response Detail Comparison")
    ax2.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.savefig("task_c_performance_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\n对比图已保存到 task_c_performance_comparison.png")

    # 打印样例回答
    print(f"\nvLLM 样例回答（前200字）:\n{vllm_result['sample']}...")
    print(f"\nagicto 样例回答（前200字）:\n{cloud_result['sample']}...")

---
## 步骤 5：模型选型讨论

下表比较不同开源模型在 GPU 服务器上的部署可行性：

| 模型 | 参数量 | 显存需求（FP16） | 中文能力 | 特点 |
|------|--------|----------------|---------|------|
| Qwen3-8B | 8B | ~16GB | 优秀 | 阿里最新模型，能力强 |
| Qwen2.5-7B-Instruct | 7B | ~14GB | 优秀 | 成熟稳定，中文最强 |
| Qwen2.5-14B-Instruct | 14B | ~28GB | 优秀 | 效果更好，需要更多显存 |
| Llama-3.2-3B-Instruct | 3B | ~6GB | 一般 | Meta 出品，轻量级 |

**RTX 3090（24GB 显存）推荐方案**：
- Qwen3-8B（FP16，约 16GB 显存）— 最新模型，效果最佳
- Qwen2.5-7B-Instruct（FP16，约 14GB 显存）— 成熟稳定
- Qwen2.5-14B-Instruct（量化到 INT8，约 14GB 显存）— 效果更好但需量化

**V100（32GB 显存）推荐方案**：
- Qwen3-8B（FP16，约 16GB 显存）— 显存充裕
- Qwen2.5-14B-Instruct（FP16，约 28GB 显存）— 显存可以支持

---
## 总结与思考题

### vLLM 部署流程回顾

```
GPU 服务器 → 安装 vLLM → 下载模型(Qwen3-8B) → 启动 OpenAI API 服务 → 本地/远程调用
```

### 实验结论

| 对比维度 | vLLM 本地部署 | agicto 云端 API |
|----------|-------------|----------------|
| 响应速度 | 取决于 GPU 性能 | 相对稳定 |
| 输出质量 | 受模型大小影响 | 云端模型通常更强 |
| 数据安全 | 数据不出服务器 | 需传输到云端 |
| 运维成本 | 需要维护 GPU 服务器 | 按量付费，免运维 |
| 扩展性 | 受限于单机性能 | 云端弹性扩缩 |

### 思考题

**1. vLLM 相比直接用 HuggingFace 推理有什么优势？**

vLLM 使用 PagedAttention 技术，显存利用率更高；支持连续批处理（Continuous Batching），吞吐量大幅提升；提供 OpenAI 兼容 API，迁移成本低，可直接用现有 SDK；内置 KV Cache 优化，推理速度比原生 HuggingFace 快 24 倍。

**2. 如果显存不够，有哪些方法可以降低显存占用？**

- **量化**：INT8/INT4 量化可将显存需求降低 50%-75%
- **选择更小的模型**：如 Qwen3-8B → Qwen2.5-3B
- **减少 max-model-len**：缩短最大序列长度
- **使用 CPU Offloading**：将部分层放到 CPU 内存
- **张量并行（Tensor Parallelism）**：多 GPU 分担

**3. 本地部署和云端 API 各有什么优劣？在实际项目中如何选择？**

- **本地部署**：数据安全可控、无网络延迟、长期成本低，但需要硬件投入和运维
- **云端 API**：开箱即用、免运维、弹性伸缩，但数据需传输到云端、按量付费
- **选择建议**：数据敏感（涉密/隐私）→ 本地部署；原型验证 / 低并发 → 云端 API；高并发 / 长期运行 → 本地部署更经济

---
任务 C 完成！